# Train the deepfake detector on Colab

Run this with a GPU: **Runtime → Change runtime type → T4 GPU**.

Before starting, on your Mac:
1. Build the face crops: `python -m model.prepare_frames --celeb-df-root ~/Documents/Celeb-DF-v2`
2. Zip them: `cd data && zip -rq faces.zip faces`
3. Upload `faces.zip` to Google Drive in a folder called `deepfake` (so it's at `MyDrive/deepfake/faces.zip`).
4. Push the repo to GitHub and paste its URL below.

Everything this notebook produces (checkpoints, metrics, plots) is saved back to `MyDrive/deepfake/`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/deepfake"
REPO_URL = "https://github.com/YOUR_USERNAME/deepfake-detector.git"  # <- change this

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q $REPO_URL /content/deepfake-detector
%cd /content/deepfake-detector
!pip install -q -r requirements-train.txt

Copy the face crops from Drive to the Colab disk. Reading thousands of small files straight from Drive is very slow.

In [ ]:
!cp "$DRIVE_DIR/faces.zip" /content/faces.zip
!unzip -q -o /content/faces.zip -d /content/
!head -3 /content/faces/manifest.csv && wc -l /content/faces/manifest.csv

## Train

ResNet50 for ~8 epochs. Each epoch prints train loss, validation loss, and validation AUC at frame and video level.
The best checkpoint (by validation AUC) is kept, and a calibration temperature is fit at the end.

In [ ]:
!python -m model.train \
    --data-dir /content/faces \
    --arch resnet50 \
    --epochs 8 \
    --batch-size 64 \
    --num-workers 2 \
    --checkpoint-path "$DRIVE_DIR/checkpoints/best_resnet50.pth" \
    --reports-dir "$DRIVE_DIR/reports/resnet50"

## Evaluate on the official Celeb-DF test set

Prints a markdown table you can paste straight into the README, plus robustness results under compression, downscaling and blur.

In [ ]:
!python -m model.eval \
    --data-dir /content/faces \
    --checkpoint-path "$DRIVE_DIR/checkpoints/best_resnet50.pth" \
    --reports-dir "$DRIVE_DIR/reports/resnet50"

In [ ]:
from IPython.display import Image, display
for name in ["training_curves.png", "roc_curve.png", "reliability_diagram.png"]:
    display(Image(f"{DRIVE_DIR}/reports/resnet50/{name}", width=520))

## Optional: compare with EfficientNet-B0

EfficientNet is the usual backbone in deepfake-detection papers. Running both gives you a real comparison for the README.

In [ ]:
!python -m model.train --data-dir /content/faces --arch efficientnet_b0 --epochs 8 --batch-size 64 --num-workers 2 \
    --checkpoint-path "$DRIVE_DIR/checkpoints/best_efficientnet_b0.pth" --reports-dir "$DRIVE_DIR/reports/efficientnet_b0"
!python -m model.eval --data-dir /content/faces --checkpoint-path "$DRIVE_DIR/checkpoints/best_efficientnet_b0.pth" \
    --reports-dir "$DRIVE_DIR/reports/efficientnet_b0"

## Next

Download the checkpoint you want to ship from `MyDrive/deepfake/checkpoints/`, rename it to `best.pth`,
and put it in `model/checkpoints/` in the repo. The API loads it from there.

Also copy the `reports/` plots into the repo's `reports/` folder so the README can show them.